In [2]:
# [셀 1] 데이터 로드 + 피처/타깃 정의 + categorical 선언
import pandas as pd
import numpy as np
import lightgbm as lgb   # pip install lightgbm scikit-learn

OUT_DIR = r'C:\Users\kjlee\도심관련\지하철관련'

train = pd.read_csv(f'{OUT_DIR}\\train.csv', index_col='ts', parse_dates=['ts'])
val   = pd.read_csv(f'{OUT_DIR}\\val.csv',   index_col='ts', parse_dates=['ts'])
test  = pd.read_csv(f'{OUT_DIR}\\test.csv',  index_col='ts', parse_dates=['ts'])

FEATURES = ['hour', 'slot', 'dow', 'month', 'is_weekend', 'is_holiday',
            'is_major_holiday', 'is_minor_holiday', 'holiday_prev', 'holiday_next',
            'is_sandwich', 'is_yearend']
TARGETS  = [c for c in train.columns if c.startswith('y_')]

X_tr, X_va, X_te = train[FEATURES].copy(), val[FEATURES].copy(), test[FEATURES].copy()

# dow/month는 '순서 있는 숫자'가 아니라 '범주' -> categorical 선언
# 트리가 {월,금} vs {화수목}, {12월,1월} vs 나머지 같은 임의 묶음 분기를 할 수 있게 됨
# (실험 결과: 정수 65.4 -> categorical 60.7, 원핫보다도 나음)
for c in ['dow', 'month']:
    X_tr[c] = X_tr[c].astype('category')
    X_va[c] = X_va[c].astype('category')
    X_te[c] = X_te[c].astype('category')

print(f'train {len(train):,} / val {len(val):,} / test {len(test):,}')
print(f'피처 {len(FEATURES)}개, 타깃 {len(TARGETS)}개')

train 126,357 / val 21,252 / test 21,252
피처 12개, 타깃 8개


In [3]:
# [셀 2] 8개 타깃 전체 학습
# 같은 문제지(X, 13피처), 정답 컬럼만 바꿔가며 모델 8개를 순차 학습

import time

models  = {}
results = []

print(f'{"타깃":14s} {"트리수":>6s} {"val MAE":>9s} {"타깃평균":>9s} {"상대오차":>8s} {"시간":>6s}')
print('-' * 62)

t_all = time.time()
for target in TARGETS:
    t0 = time.time()

    m = lgb.LGBMRegressor(
        n_estimators=2000, learning_rate=0.05, num_leaves=31,
        random_state=42, verbose=-1,
    )
    m.fit(
        X_tr, train[target],
        eval_set=[(X_va, val[target])],       # 매 트리마다 val 채점
        eval_metric='mae',
        callbacks=[lgb.early_stopping(100, verbose=False)],  # 100트리 개선 없으면 중단
    )

    mae  = np.abs(m.predict(X_va) - val[target]).mean()
    mean = val[target].mean()
    models[target] = m
    results.append({'target': target, 'trees': m.best_iteration_,
                    'val_MAE': round(mae, 1), 'rel_err': round(mae / mean * 100, 1)})
    print(f'{target:14s} {m.best_iteration_:>6d} {mae:>8.1f}명 {mean:>8.1f}명 {mae/mean*100:>7.1f}% {time.time()-t0:>5.1f}s')

print('-' * 62)
print(f'전체 완료: {time.time() - t_all:.1f}초')
results = pd.DataFrame(results)

타깃                트리수   val MAE      타깃평균     상대오차     시간
--------------------------------------------------------------
y_board_5         803     11.1명     51.4명    21.6%   3.1s
y_board_15        688     22.9명    154.1명    14.9%   1.5s
y_board_30        586     36.7명    308.3명    11.9%   1.2s
y_board_60        303     61.8명    616.7명    10.0%   0.7s
y_alight_5       2000      9.5명     48.7명    19.5%   3.5s
y_alight_15      1362     18.4명    146.1명    12.6%   2.5s
y_alight_30       629     30.8명    292.2명    10.6%   1.3s
y_alight_60       465     55.4명    583.4명     9.5%   1.0s
--------------------------------------------------------------
전체 완료: 14.8초


In [4]:
# [셀 3] test 최종 채점 — 전체 / 평상일 / 특이일 분리
# 확장 피처의 효과는 특이일에 집중되므로 뭉뚱그린 MAE만 보면 개선이 가려짐

# 특이일 정의: 공휴일이거나, 공휴일 앞뒤이거나, 샌드위치·연말인 날
special = ((test.is_holiday == 1) | (test.holiday_prev == 1) | (test.holiday_next == 1)
           | (test.is_sandwich == 1) | (test.is_yearend == 1))

final = []
for target in TARGETS:
    pred = models[target].predict(X_te)
    err  = np.abs(pred - test[target])
    final.append({
        'target': target,
        'MAE_전체':   round(err.mean(), 1),
        'MAE_평상일': round(err[~special].mean(), 1),
        'MAE_특이일': round(err[special].mean(), 1),
        '상대오차':   f'{err.mean() / test[target].mean() * 100:.1f}%',
    })

final = pd.DataFrame(final)
print(f'특이일 비중: {special.mean():.0%} ({special.sum():,}행)')
final

특이일 비중: 18% (3,927행)


,target,MAE_전체,MAE_평상일,MAE_특이일,상대오차
0,y_board_5,10.9,10.8,11.7,21.7%
1,y_board_15,22.4,21.3,27.2,14.8%
2,y_board_30,35.5,33.3,45.4,11.7%
3,y_board_60,61.2,55.8,84.9,10.1%
4,y_alight_5,9.4,8.9,11.8,19.5%
5,y_alight_15,18.9,17.6,24.5,13.0%
6,y_alight_30,31.8,29.3,42.9,11.0%
7,y_alight_60,57.5,52.3,80.6,9.9%


In [5]:
# [셀 5] 모델 저장 (models 폴더에)
import joblib, os

MODEL_DIR = f'{OUT_DIR}\\models'
os.makedirs(MODEL_DIR, exist_ok=True)   # 폴더 없으면 생성, 있으면 그냥 통과

MODEL_PATH = f'{MODEL_DIR}\\models_v3.pkl'
joblib.dump(models, MODEL_PATH)
print('저장:', MODEL_PATH)

# 복원할 때:
# models = joblib.load(MODEL_PATH)

저장: C:\Users\kjlee\도심관련\지하철관련\models\models_v3.pkl


In [ ]:
# [셀 6] 원샷 예측 함수: 날짜·시간 문자열 → 승하차 8개 예측
# 사용법: predict_at('2026-08-15 14:20')
import holidays

# 공휴일 달력 준비 (미래 조회 대비 2030년까지)
_kr = holidays.KR(years=range(2024, 2031), language='ko')
_hol = pd.to_datetime(list(_kr.keys()))
_major = pd.to_datetime([d for d, n in _kr.items() if ('설날' in n) or ('추석' in n)])

def _off(day):   # 그 날이 쉬는 날인가 (공휴일 or 주말)
    return (day in _hol) or (day.dayofweek >= 5)

def predict_at(ts_str):
    t = pd.Timestamp(ts_str)
    day, prev, nxt = t.normalize(), t.normalize() - pd.Timedelta('1D'), t.normalize() + pd.Timedelta('1D')

    # 전처리 셀 7~8과 동일한 로직으로 피처 13개 생성 (날짜만으로 전부 계산됨)
    x = pd.DataFrame([{
        'hour': t.hour,
        'slot': t.hour * 12 + t.minute // 5,
        'dow': t.dayofweek,
        'month': t.month,
        'is_weekend': int(t.dayofweek >= 5),
        'is_holiday': int(day in _hol),
        'is_major_holiday': int(day in set(_major)),
        'is_minor_holiday': int((day in _hol) and (day not in set(_major))),
        'holiday_prev': int(prev in _hol),
        'holiday_next': int(nxt in _hol),
        'is_sandwich': int((not _off(day)) and _off(prev) and _off(nxt)),
        'is_yearend': int(t.month == 12 and t.day >= 24),
    }])[FEATURES]

    # ★ categorical은 학습 때와 같은 범주 목록으로 선언해야 함 (한 줄짜리 데이터라 명시 필수)
    x['dow']   = pd.Categorical(x['dow'],   categories=range(7))
    x['month'] = pd.Categorical(x['month'], categories=range(1, 13))

    # 모델 8개 예측 + 단조성 보정(5≤15≤30≤60)
    out = {}
    for col in ['board', 'alight']:
        run = 0
        for w in [5, 15, 30, 60]:
            run = max(run, float(models[f'y_{col}_{w}'].predict(x)[0]))
            out[f'{col}_{w}'] = round(run)
    return out

# 테스트
for ts in ['2025-12-24 18:20']:
    r = predict_at(ts)
    print(f'{ts} → 승차 {r["board_5"]}/{r["board_15"]}/{r["board_30"]}/{r["board_60"]}명, '
          f'하차 {r["alight_5"]}/{r["alight_15"]}/{r["alight_30"]}/{r["alight_60"]}명 (+5/15/30/60분)')